In [1]:
import sys
import os
sys.path.append(os.getcwd())
# Fix path for notebook
notebook_dir = os.path.abspath('.')
if notebook_dir not in sys.path:
    sys.path.append(notebook_dir)

import pandas as pd
from datetime import datetime
import time

# Now imports will work
import config
from src.data_engine import DataRepository
from src.features import FeatureEngineer
from src.strategy import SEPAStrategy
from src.database import DatabaseManager

In [2]:
data_repo = DataRepository()
db = DatabaseManager()
tickers = data_repo.update_universe()
results = data_repo.update_cache(tickers, force=False, source='fmp')

ticker_data = data_repo.get_batch_data(tickers)
valid_ticker_data = {
    ticker: df for ticker, df in ticker_data.items() 
    if df is not None and len(df) >= 200
}

benchmark_data = data_repo.get_benchmark_data()
feature_engine = FeatureEngineer(benchmark_data=benchmark_data)
strategy = SEPAStrategy(benchmark_data=benchmark_data)

enriched_data = feature_engine.process_universe_batch(valid_ticker_data)
results = strategy.batch_scan_universe(enriched_data, scan_date=None)


INFO:src.database:Database initialized at c:\Users\Hang\PycharmProjects\quantamental\database\trades.db
INFO:src.data_engine:Fetching S&P 500 universe from SSGA...
INFO:src.data_engine:Successfully loaded 504 unique tickers
INFO:src.data_engine:Updating cache for 2/504 tickers...
INFO:src.data_engine:Using FMP API for batch update...
INFO:src.data_engine:Processing FMP batch 1/1 (2 tickers)
ERROR:src.data_engine:FMP API request failed: 403 Client Error: Forbidden for url: https://financialmodelingprep.com/api/v3/historical-price-full/RE,-?apikey=ijhOTOkHXfjKurcgD8X3ENFPjrnR5MPh
INFO:src.data_engine:FMP batch update: 0/2 successful
ERROR:src.data_engine:FMP API request failed: 403 Client Error: Forbidden for url: https://financialmodelingprep.com/api/v3/historical-price-full/RE?apikey=ijhOTOkHXfjKurcgD8X3ENFPjrnR5MPh
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['RE']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol ma

In [ ]:
qualifying_stocks = results['qualifying_stocks']
new_triggers_today = results['new_triggers']
actual_latest_date = results['summary']['latest_date']
